# Imputation Environment Check

Run the next cell to verify that required libraries are available for:
- Mean imputation (`SimpleImputer`)
- kNN imputation (`KNNImputer`)
- MICE and SoftImpute workflows (via `hyperimpute`)
- Research repos (`GRAPE`, `DiffPuter`)

In [1]:
import sys
import pathlib
import importlib
from importlib import metadata

# ---- self-contained DiffPuter path setup ----------------------------------
# This makes the cell work on its own. If you also have a separate path-setup
# cell, that's fine — running it twice is a no-op.
HERE = pathlib.Path.cwd()
for cand in [
    HERE / "DiffPuter",
    HERE / "external" / "DiffPuter",
    HERE.parent / "DiffPuter",
]:
    if (cand / "main.py").is_file():
        for p in [cand, cand / "baselines", cand / "baselines" / "GRAPE"]:
            sp = str(p.resolve())
            if sp not in sys.path:
                sys.path.insert(0, sp)
        break

# Module name (for import) -> distribution name (for version lookup).
# These differ for some packages (sklearn vs scikit-learn, yaml vs PyYAML,
# ot vs POT).
required_modules = {
    # core scientific
    "numpy":         "numpy",
    "pandas":        "pandas",
    "scipy":         "scipy",
    "sklearn":       "scikit-learn",
    "matplotlib":    "matplotlib",
    "statsmodels":   "statsmodels",

    # baselines + SOTA
    "fancyimpute":   "fancyimpute",     # Mean (SimpleFill), kNN, SoftImpute
    "hyperimpute":   "hyperimpute",     # HyperImpute + MICE plugin

    # deep learning + GRAPE deps
    "torch":             "torch",
    "torch_geometric":   "torch_geometric",
    "h5py":              "h5py",
    "networkx":          "networkx",

    # DiffPuter deps
    "ot":            "POT",             # POT distributes as POT, imports as ot
    "FrEIA":         "FrEIA",
    "timm":          "timm",
    "yaml":          "PyYAML",

    # notebook
    "ipykernel":     "ipykernel",
}

# Modules that are nice-to-have but the env still works without them.
optional_modules = {
    "torch_scatter": "torch_scatter",   # GRAPE only
}

# Note: GRAPE and DiffPuter are loaded as local source clones, not pip packages.


def check_module(module_name: str, package_name: str):
    try:
        importlib.import_module(module_name)
    except Exception as e:
        return "MISSING", str(e)
    try:
        version = metadata.version(package_name)
    except metadata.PackageNotFoundError:
        version = "unknown"
    return "OK", version


print("Required packages")
print("-" * 60)
missing = []
for module_name, package_name in required_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "✗"
    print(f"  {marker} {package_name:18} {status:8} {info}")
    if status == "MISSING":
        missing.append(package_name)

print("\nOptional packages")
print("-" * 60)
for module_name, package_name in optional_modules.items():
    status, info = check_module(module_name, package_name)
    marker = "✓" if status == "OK" else "○"
    print(f"  {marker} {package_name:18} {status:8} {info}")

print("\nSmoke-test imports for imputation APIs")
print("-" * 60)

api_checks = [
    ("Mean / kNN (sklearn)",
     lambda: __import__("sklearn.impute", fromlist=["SimpleImputer", "KNNImputer"])),
    ("MICE (sklearn IterativeImputer)",
     lambda: (
         __import__("sklearn.experimental", fromlist=["enable_iterative_imputer"]),
         __import__("sklearn.impute",       fromlist=["IterativeImputer"]),
     )),
    ("SoftImpute / KNN / SimpleFill (fancyimpute)",
     lambda: __import__("fancyimpute", fromlist=["SoftImpute", "KNN", "SimpleFill"])),
    ("HyperImpute (hyperimpute.plugins.imputers.Imputers)",
     lambda: __import__("hyperimpute.plugins.imputers", fromlist=["Imputers"])),
    ("PyTorch Geometric (for GRAPE)",
     lambda: __import__("torch_geometric")),
]

for label, fn in api_checks:
    try:
        fn()
        print(f"  ✓ {label}: OK")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {e}")

print("\nLocal repo modules (DiffPuter / GRAPE)")
print("-" * 60)
repo_checks = [
    ("DiffPuter dataset",   "dataset",          ["load_dataset", "get_eval", "mean_std"]),
    ("DiffPuter diffusion", "diffusion_utils",  ["sample_step", "impute_mask", "EDMLoss"]),
    ("DiffPuter model",     "model",            ["MLPDiffusion", "Model"]),
    ("GRAPE training",      "training.gnn_mdi", ["train_gnn_mdi"]),
]
for label, mod, names in repo_checks:
    try:
        m = importlib.import_module(mod)
        for n in names:
            getattr(m, n)
        print(f"  ✓ {label}: OK ({mod})")
    except Exception as e:
        print(f"  ✗ {label}: FAILED | {type(e).__name__}: {e}")

print()
if missing:
    print(f"⚠  Missing required packages: {', '.join(missing)}")
    print("   Re-run: pip install -r requirements-base.txt")
else:
    print("All required packages present.")

Required packages
------------------------------------------------------------
  ✓ numpy              OK       1.26.4
  ✓ pandas             OK       2.3.3
  ✓ scipy              OK       1.17.1
  ✓ scikit-learn       OK       1.6.1
  ✓ matplotlib         OK       3.10.9
  ✓ statsmodels        OK       0.14.6
  ✓ fancyimpute        OK       0.7.0
  ✓ hyperimpute        OK       0.1.17
  ✓ torch              OK       2.4.1
  ✓ torch_geometric    OK       2.7.0
  ✓ h5py               OK       3.14.0
  ✓ networkx           OK       3.6.1


I0000 00:00:1778612111.841418    4983 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


  ✓ POT                OK       0.9.6.post1
  ✓ FrEIA              OK       0.2
  ✓ timm               OK       1.0.27
  ✓ PyYAML             OK       6.0.3
  ✓ ipykernel          OK       7.2.0

Optional packages
------------------------------------------------------------
  ✓ torch_scatter      OK       2.1.2+pt24cu121

Smoke-test imports for imputation APIs
------------------------------------------------------------
  ✓ Mean / kNN (sklearn): OK
  ✓ MICE (sklearn IterativeImputer): OK
  ✓ SoftImpute / KNN / SimpleFill (fancyimpute): OK
  ✓ HyperImpute (hyperimpute.plugins.imputers.Imputers): OK
  ✓ PyTorch Geometric (for GRAPE): OK

Local repo modules (DiffPuter / GRAPE)
------------------------------------------------------------
  ✓ DiffPuter dataset: OK (dataset)
  ✓ DiffPuter diffusion: OK (diffusion_utils)
  ✓ DiffPuter model: OK (model)
  ✓ GRAPE training: OK (training.gnn_mdi)

All required packages present.


# Dataset Analysis
Perform simple data exploration: Shape, Columns, Data types, Missing values, First few rows

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Load the dataset
df = pd.read_csv('Scenario5/scenario5.csv')

# Drop unnamed columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Convert relative paths to absolute for reference
base_dir = Path('Scenario5')

# Data exploration
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst few rows:")
df.head()

Shape: (2300, 15)

Columns: ['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit1_beam_index', 'seq_index', 'time_stamp[UTC]', 'unit2_direction', 'unit2_num_sat', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS', 'unit2_PDOP', 'unit2_HDOP']

Data types:
index                 int64
unit1_rgb            object
unit1_pwr_60ghz      object
unit1_loc            object
unit2_loc            object
unit1_beam_index      int64
seq_index             int64
time_stamp[UTC]      object
unit2_direction       int64
unit2_num_sat         int64
unit2_sat_used       object
unit2_fix_type       object
unit2_DGPS           object
unit2_PDOP          float64
unit2_HDOP          float64
dtype: object

Missing values:
index               0
unit1_rgb           0
unit1_pwr_60ghz     0
unit1_loc           0
unit2_loc           0
unit1_beam_index    0
seq_index           0
time_stamp[UTC]     0
unit2_direction     0
unit2_num_sat       0
unit2_sat_used      0
unit2_fix_type      0
unit2_DGPS

,index,unit1_rgb,unit1_pwr_60ghz,unit1_loc,unit2_loc,unit1_beam_index,seq_index,time_stamp[UTC],unit2_direction,unit2_num_sat,unit2_sat_used,unit2_fix_type,unit2_DGPS,unit2_PDOP,unit2_HDOP
0,1,./unit1/camera_data/image_BS1_259_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_0.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_0.txt,60,1,['03-20-31-142'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
1,2,./unit1/camera_data/image_BS1_260_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_1.txt,60,1,['03-20-31-284'],1,24,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
2,3,./unit1/camera_data/image_BS1_261_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_2.txt,58,1,['03-20-31-426'],1,14,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
3,4,./unit1/camera_data/image_BS1_262_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_3.txt,59,1,['03-20-31-568'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6
4,5,./unit1/camera_data/image_BS1_263_03_20_31.jpg,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_4.txt,60,1,['03-20-31-710'],1,21,G2 G5 G12 G18 G25 G29 R5 R6 R7 R10 R12 R20 R21...,3D,Yes,1.1,0.6


# Dataset preprocessing
- Convert path columns into value columns
- Drop redundant/non-usable columns

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

# Parse location .txt files into lat/lon columns
base_dir = Path('Scenario5')

def parse_loc_file(rel_path):
    """
    Read a location .txt file and return (lat, lon) as floats. Handles both comma- and whitespace-separated formats.
    """
    if pd.isna(rel_path):
        return np.nan, np.nan
    try:
        full_path = base_dir / rel_path if not Path(rel_path).is_absolute() else Path(rel_path)
        with open(full_path, 'r') as f:
            content = f.read().strip()
        # Handle comma- or whitespace-separated
        parts = [p.strip() for p in content.replace(',', ' ').split()]
        lat, lon = float(parts[0]), float(parts[1])
        return lat, lon
    except (FileNotFoundError, ValueError, IndexError) as e:
        return np.nan, np.nan

# Peek at one file first to confirm format before parsing all 2300
sample_path = base_dir / df['unit1_loc'].iloc[0]
print(f"Sample loc file ({sample_path}):")
with open(sample_path, 'r') as f:
    print(repr(f.read()))

df[['unit2_lat', 'unit2_lon']] = df['unit2_loc'].apply(
    lambda p: pd.Series(parse_loc_file(p))
)

print(f"\nParsed coordinates — first few rows:")
print(df[['unit2_lat', 'unit2_lon']].head())
print(f"\nParse failures (NaNs) per column:")
print(df[['unit2_lat', 'unit2_lon']].isna().sum())

Sample loc file (Scenario5/unit1/GPS_data/gps_location.txt):
'33.42069305555555\n-111.929175\n\n\n'

Parsed coordinates — first few rows:
   unit2_lat   unit2_lon
0  33.420483 -111.928924
1  33.420488 -111.928924
2  33.420492 -111.928924
3  33.420497 -111.928925
4  33.420502 -111.928925

Parse failures (NaNs) per column:
unit2_lat    0
unit2_lon    0
dtype: int64


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

base_dir = Path('Scenario5')
N_BEAMS = 64

def parse_pwr_file(rel_path):
    """
    Read an mmWave power .txt file and return a length-64 numpy array.
    Each line is one beam's received power (scientific notation).
    Returns array of NaNs on failure.
    """
    if pd.isna(rel_path):
        return np.full(N_BEAMS, np.nan)
    try:
        full_path = base_dir / rel_path if not Path(rel_path).is_absolute() else Path(rel_path)
        vals = np.loadtxt(full_path)
        if vals.shape != (N_BEAMS,):
            # Unexpected shape — flag it rather than silently truncating
            return np.full(N_BEAMS, np.nan)
        return vals
    except (FileNotFoundError, ValueError):
        return np.full(N_BEAMS, np.nan)

# Parse all files into a (n_rows, 64) array — fast vectorised assignment
print("Parsing mmWave power files...")
pwr_matrix = np.vstack([parse_pwr_file(p) for p in df['unit1_pwr_60ghz']])
print(f"Power matrix shape: {pwr_matrix.shape}")
print(f"Files that failed to parse: {np.isnan(pwr_matrix).all(axis=1).sum()}")

# Expand into 64 columns: beam_00, beam_01, ..., beam_63
beam_cols = [f'beam_{i:02d}' for i in range(N_BEAMS)]
pwr_df = pd.DataFrame(pwr_matrix, columns=beam_cols, index=df.index)

# Check if the beam with max power matches the recorded optimal beam index (unit1_beam_index)
parsed_argmax = np.argmax(pwr_matrix, axis=1) + 1
match_rate = (parsed_argmax == df['unit1_beam_index'].values).mean()
print(f"Argmax matches unit1_beam_index: {match_rate*100:.2f}%")

# Build clean tabular feature set: 64 beams + GPS quality + receiver location
df_add_pwr = pd.concat([
    pwr_df,                                                                  # 64 beam columns
    df
], axis=1)

print(f"\nFinal feature shape: {df_add_pwr.shape}")
print(f"\nMissing values before injection: {df_add_pwr.isna().sum().sum()}")

Parsing mmWave power files...
Power matrix shape: (2300, 64)
Files that failed to parse: 0
Argmax matches unit1_beam_index: 100.00%

Final feature shape: (2300, 81)

Missing values before injection: 0


### Dropped columns

`index` — just a row counter (1, 2, 3...). Perfectly correlated with row position, carries no information.

`unit1_rgb` — image file paths. Not tabular data. Visual features would require a separate embedding pipeline and are out of scope for tabular imputation.

`unit1_pwr_60ghz` — original file paths to mmWave power readings. Replaced by 64 parsed `beam_XX` columns containing the actual per-beam received power values.

`unit1_loc` — paths to GPS files for the static basestation. Unit1 is stationary, so after parsing the coordinates are constant across all rows. A constant column has zero variance and zero predictive value; including it would also give imputers a free 100% accuracy on that column, artificially inflating overall metrics.

`unit2_loc` — original file paths for the mobile receiver's GPS readings. Replaced by parsed `unit2_lat` and `unit2_lon` columns.

`unit1_beam_index` — the optimal beam index, computed as `argmax(beam_00..beam_63) + 1`. Including it alongside the 64 beam columns would be perfect leakage: any model could trivially recover it from the beam values. Held aside as the evaluation target (`y_optimal_beam`) for downstream "did imputation preserve the optimal beam?" analysis.

`seq_index` — sequence identifier; rows belonging to the same trajectory share a value. Metadata for grouping rather than a feature. Dropped for imputation experiments but will be retained separately for sequence-aware train/test splits in downstream modelling.

`time_stamp[UTC]` — within-second timestamp in `'03-20-31-142'` format. Not directly usable as a tabular feature without further engineering (e.g. time-of-day, inter-row deltas), and not needed for the missingness experiments.

`unit2_sat_used` — string-encoded list of locked satellite IDs (e.g. "G2 G5 G12 G18 G25 G29 R5 R6..."). Tabular use would require one-hot encoding every possible satellite (dozens of columns) or a count. We already have `unit2_num_sat` as the count, making this column redundant.

`unit2_fix_type` — constant value (`"3D"`) across all rows. Zero variance, no information.

`unit2_DGPS` — constant value (`"Yes"`) across all rows. Zero variance, no information.

### Retained columns

64 `beam_00`..`beam_63` columns — per-beam received power at 60 GHz. The core signal for 6G beam-prediction work. Adjacent beams are spatially correlated (smooth angular response with a main lobe), which gives correlation-aware imputers a meaningful edge over naive ones.

`unit2_lat`, `unit2_lon` — parsed coordinates of the mobile receiver. Vary across rows and correlate with which beam is geometrically optimal.

`unit2_direction` — direction of motion (0 or 1). Binary but non-constant; informative spatial context.

`unit2_num_sat`, `unit2_PDOP`, `unit2_HDOP` — GPS quality indicators reflecting satellite visibility and geometric dilution of precision. Useful as side information for imputation, since poor GPS conditions tend to correlate with poor mmWave link quality (multipath, obstructions).

In [5]:
print (df_add_pwr['unit2_DGPS'].nunique()) # Drop unit2_DGPS since it has only one unique value and thus provides no useful information for imputation.
print(df_add_pwr['unit2_fix_type'].nunique()) # Drop unit2_fix_type since it has only one unique value and thus provides no useful information for imputation.

print(df.columns)

drop_columns = ['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit1_beam_index','seq_index', 'time_stamp[UTC]', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS']

clean_data = df_add_pwr.drop(columns=drop_columns)

print("Remaining columns in clean_data:")
print(clean_data.columns)
print(clean_data)

# Total missing values after parsing but before injection
total_missing = clean_data.isna().sum().sum()
print(f"\nTotal missing values after parsing: {total_missing}")

1
1
Index(['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc',
       'unit1_beam_index', 'seq_index', 'time_stamp[UTC]', 'unit2_direction',
       'unit2_num_sat', 'unit2_sat_used', 'unit2_fix_type', 'unit2_DGPS',
       'unit2_PDOP', 'unit2_HDOP', 'unit2_lat', 'unit2_lon'],
      dtype='object')
Remaining columns in clean_data:
Index(['beam_00', 'beam_01', 'beam_02', 'beam_03', 'beam_04', 'beam_05',
       'beam_06', 'beam_07', 'beam_08', 'beam_09', 'beam_10', 'beam_11',
       'beam_12', 'beam_13', 'beam_14', 'beam_15', 'beam_16', 'beam_17',
       'beam_18', 'beam_19', 'beam_20', 'beam_21', 'beam_22', 'beam_23',
       'beam_24', 'beam_25', 'beam_26', 'beam_27', 'beam_28', 'beam_29',
       'beam_30', 'beam_31', 'beam_32', 'beam_33', 'beam_34', 'beam_35',
       'beam_36', 'beam_37', 'beam_38', 'beam_39', 'beam_40', 'beam_41',
       'beam_42', 'beam_43', 'beam_44', 'beam_45', 'beam_46', 'beam_47',
       'beam_48', 'beam_49', 'beam_50', 'beam_51', 'beam_52', 'beam_5

# Amputation experiments: MCAR, MAR, and MNAR

All three mechanisms target the same set of columns — the 64 beam-power columns — at 10%, 30%, and 50% missingness. Keeping the target columns fixed across mechanisms means any difference in RMSE/MAE between scenarios reflects the **mechanism**, not a different denominator. GPS columns remain fully observed throughout and act as side information (and as drivers for MAR).

The three missingness mechanisms, introduced by Rubin (1976), formalise *what the probability of missingness depends on*. Let $Y$ denote the full data, $Y_\text{obs}$ the observed part, $Y_\text{mis}$ the missing part, and $M$ the missingness indicator.

**MCAR — Missing Completely At Random.** $P(M \mid Y) = P(M)$. Missingness is independent of all data values, observed or unobserved. *Our implementation:* each beam cell is masked independently with probability $p$ (Bernoulli draws). This is the baseline — the "no-signal" case where mean imputation is only beaten by methods that exploit cross-column structure.

**MAR — Missing At Random.** $P(M \mid Y) = P(M \mid Y_\text{obs})$. Missingness may depend on observed values but, conditional on those, is independent of the missing ones. MAR is the standard working assumption for imputation methods — under MAR, likelihood-based and Bayesian inference can ignore the missingness mechanism (Rubin 1976; Little & Rubin 2019). *Our implementation:* per-row missingness probability is a logistic function of the observed, standardised GPS-quality drivers (`unit2_PDOP + unit2_HDOP`), with the intercept calibrated per call so the marginal beam-missingness rate matches $p$. Physical story: degraded satellite geometry tends to coincide with poorer mmWave links, so beam measurements drop out more often when GPS is poor.

**MNAR — Missing Not At Random.** $P(M \mid Y)$ still depends on $Y_\text{mis}$ even after conditioning on $Y_\text{obs}$. The classic example is self-masking: a value's probability of being missing depends on the value itself (Little & Rubin 2019; van Buuren 2018). MNAR is non-ignorable — standard imputers will be biased in principle, and recovery is fundamentally harder. *Our implementation:* each beam cell is masked with probability that's a logistic function of its own (standardised) value, with negative slope so that **weaker beams are more likely to be dropped**. Physical story: low-power beams sit closer to the receiver's noise floor and are more likely to be thresholded out. The slope and intercept are calibrated per call so the marginal rate matches $p$.

For each scenario, every (proportion × method) cell is averaged over multiple seeds. Metrics reported: **RMSE** and **MAE** on the amputed cells.

**References.**

- Rubin, D. B. (1976). Inference and missing data. *Biometrika* 63(3), 581–592.
- Little, R. J. A., & Rubin, D. B. (2019). *Statistical Analysis with Missing Data* (3rd ed.). Wiley.
- van Buuren, S. (2018). *Flexible Imputation of Missing Data* (2nd ed.). Chapman & Hall/CRC.

In [6]:
# Import libraries for experiment driver and external imputers module

import time

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from imputers.imputers import (
    Imputer,
    MeanImputer,
    KNNImputerWrapper,
    MICEImputer,
    SoftImputeWrapper,
    HyperImputeImputer,
    get_default_imputers,
 )

from imputers.diffputer_imputer import DiffPuterImputer
from imputers.grape_imputer import GRAPEImputer

In [7]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

PROPORTIONS = [0.10, 0.30, 0.50]
N_SEEDS = 10

BEAM_COLS = [f"beam_{i:02d}" for i in range(64)]
GPS_COLS = [
    "unit2_lat",
    "unit2_lon",
    "unit2_direction",
    "unit2_num_sat",
    "unit2_PDOP",
    "unit2_HDOP",
]
RETAINED_COLS = BEAM_COLS + GPS_COLS

# Observed columns whose values drive MAR missingness in the beam columns.
# These stay fully observed under MAR so the mechanism remains MAR (not MNAR).
MAR_DRIVER_COLS = ["unit2_PDOP", "unit2_HDOP"]


# ---------------------------------------------------------------------------
# Sanity check
# ---------------------------------------------------------------------------

def validate_dataframe(df: pd.DataFrame) -> None:
    missing = [c for c in RETAINED_COLS if c not in df.columns]
    if missing:
        raise KeyError(
            f"clean_data is missing {len(missing)} expected column(s): "
            f"{missing[:5]}{'...' if len(missing) > 5 else ''}"
        )
    if df[RETAINED_COLS].isna().any().any():
        raise ValueError(
            "clean_data already contains NaN values in retained columns. "
            "Amputation experiments require fully observed input."
        )


# ---------------------------------------------------------------------------
# Amputation mechanisms — all three target the same columns (beams).
# ---------------------------------------------------------------------------

def _calibrate_intercept(scores: np.ndarray, slope: float, target_rate: float) -> float:
    """Bisection-solve for intercept b so mean(sigmoid(slope*scores + b)) == target_rate.

    Used by MAR (scores are row-level driver sums) and MNAR (scores are
    per-cell beam values, flattened). Works for any input shape since we
    only need the global mean.
    """
    flat = scores.reshape(-1)

    def mean_p(b):
        return float(np.mean(1.0 / (1.0 + np.exp(-(slope * flat + b)))))

    lo, hi = -50.0, 50.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if mean_p(mid) < target_rate:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def ampute_mcar(shape: tuple[int, int], prop: float,
                target_col_idx: np.ndarray,
                rng: np.random.Generator) -> np.ndarray:
    """MCAR: each target-column cell masked independently with probability `prop`."""
    mask = np.zeros(shape, dtype=bool)
    sub_mask = rng.random((shape[0], len(target_col_idx))) < prop
    mask[:, target_col_idx] = sub_mask
    return mask


def ampute_mar(data_scaled: np.ndarray, prop: float,
               target_col_idx: np.ndarray,
               driver_col_idx: np.ndarray,
               rng: np.random.Generator) -> np.ndarray:
    """MAR: per-row miss probability is a logistic function of *observed* drivers.

    Rows with higher driver score (e.g. worse GPS) are more likely to have
    their target cells masked. Driver columns themselves are never masked,
    preserving the MAR (not MNAR) condition.
    """
    n_rows = data_scaled.shape[0]
    mask = np.zeros(data_scaled.shape, dtype=bool)

    # Row-level score from standardised driver columns
    score = data_scaled[:, driver_col_idx].sum(axis=1)
    score = (score - score.mean()) / (score.std() + 1e-8)

    slope = 2.0
    intercept = _calibrate_intercept(score, slope, prop)
    p_row = 1.0 / (1.0 + np.exp(-(slope * score + intercept)))  # shape (n_rows,)

    draws = rng.random((n_rows, len(target_col_idx)))
    sub_mask = draws < p_row[:, None]
    mask[:, target_col_idx] = sub_mask
    return mask


def ampute_mnar(data_scaled: np.ndarray, prop: float,
                target_col_idx: np.ndarray,
                rng: np.random.Generator) -> np.ndarray:
    """MNAR (self-masked): each cell's miss probability depends on its own value.

    For beam-power data: weaker beams are closer to the receiver's noise
    floor and more likely to be dropped, so we use a *negative* slope —
    small (more-negative-standardised) values get higher miss probability.
    The intercept is calibrated per call so the marginal miss rate equals
    `prop`. The mechanism is MNAR because missingness on a cell depends on
    that cell's value, even given the observed data.
    """
    mask = np.zeros(data_scaled.shape, dtype=bool)

    # Per-cell scores: just the standardised beam values themselves.
    cell_vals = data_scaled[:, target_col_idx]  # shape (n_rows, n_targets)

    slope = -2.0  # negative: low values -> high miss probability
    intercept = _calibrate_intercept(cell_vals, slope, prop)
    p_cell = 1.0 / (1.0 + np.exp(-(slope * cell_vals + intercept)))

    draws = rng.random(cell_vals.shape)
    sub_mask = draws < p_cell
    mask[:, target_col_idx] = sub_mask
    return mask


# ---------------------------------------------------------------------------
# Metrics + per-run evaluation. Only beam columns are ever masked, so
# global == beams here; we report a single RMSE and MAE per run.
# ---------------------------------------------------------------------------

def evaluate_one_run(data_scaled: np.ndarray, mask: np.ndarray,
                     imputer: Imputer, seed: int) -> dict:
    """Apply mask -> impute -> compute RMSE and MAE on amputed cells."""
    data_masked = data_scaled.copy()
    data_masked[mask] = np.nan

    data_imputed = imputer.fit_transform(data_masked, seed=seed)

    # Some imputers occasionally leave NaNs; fall back to column means.
    if np.isnan(data_imputed).any():
        col_means = np.nanmean(data_masked, axis=0)
        col_means = np.where(np.isnan(col_means), 0.0, col_means)
        nan_pos = np.isnan(data_imputed)
        data_imputed = data_imputed.copy()
        data_imputed[nan_pos] = np.take(col_means, np.where(nan_pos)[1])

    err = data_scaled[mask] - data_imputed[mask]
    rmse = float(np.sqrt(np.mean(err ** 2))) if err.size else np.nan
    mae = float(np.mean(np.abs(err))) if err.size else np.nan

    return {
        "rmse": rmse,
        "mae": mae,
        "n_masked_cells": int(mask.sum()),
    }


# ---------------------------------------------------------------------------
# Main experiment driver
# ---------------------------------------------------------------------------

def run_experiments(clean_data: pd.DataFrame,
                    imputers: list[Imputer] | None = None,
                    scenario_filter: list[str] | None = None,
                    verbose: bool = True) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run all (scenario x proportion x method x seed) combinations.

    One scenario per mechanism, all targeting the 64 beam columns:
      - "MCAR": each beam cell masked independently w.p. `prop`.
      - "MAR":  row-level miss probability driven by observed GPS quality.
      - "MNAR": per-cell miss probability driven by the cell's own value.
    """
    validate_dataframe(clean_data)

    if imputers is None:
        imputers = get_default_imputers()

    # Standardise once on full clean data (consistent across all conditions).
    data_full = clean_data[RETAINED_COLS].to_numpy(dtype=float)
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data_full)

    col_to_idx = {c: i for i, c in enumerate(RETAINED_COLS)}
    beam_idx = np.array([col_to_idx[c] for c in BEAM_COLS])
    mar_driver_idx = np.array([col_to_idx[c] for c in MAR_DRIVER_COLS])

    scenarios = [
        ("MCAR", "mcar"),
        ("MAR",  "mar"),
        ("MNAR", "mnar"),
    ]

    if scenario_filter is not None:
        scenarios = [s for s in scenarios if s[0] in scenario_filter]

    total_runs = len(scenarios) * len(PROPORTIONS) * len(imputers) * N_SEEDS
    if verbose:
        print(f"Total runs: {total_runs} "
              f"({len(scenarios)} scenarios x {len(PROPORTIONS)} props "
              f"x {len(imputers)} methods x {N_SEEDS} seeds)")

    records = []
    run_counter = 0
    t_start = time.time()

    for scenario_name, mechanism in scenarios:
        for prop in PROPORTIONS:
            for imputer in imputers:
                method_t0 = time.time()
                for seed in range(N_SEEDS):
                    rng = np.random.default_rng(seed)
                    if mechanism == "mcar":
                        mask = ampute_mcar(
                            data_scaled.shape, prop, beam_idx, rng,
                        )
                    elif mechanism == "mar":
                        mask = ampute_mar(
                            data_scaled, prop, beam_idx, mar_driver_idx, rng,
                        )
                    elif mechanism == "mnar":
                        mask = ampute_mnar(
                            data_scaled, prop, beam_idx, rng,
                        )
                    else:
                        raise ValueError(f"Unknown mechanism: {mechanism}")

                    metrics = evaluate_one_run(
                        data_scaled, mask, imputer, seed,
                    )
                    records.append({
                        "scenario": scenario_name,
                        "proportion": prop,
                        "method": imputer.name,
                        "seed": seed,
                        **metrics,
                    })
                    run_counter += 1

                if verbose:
                    elapsed = time.time() - method_t0
                    print(f"  [{run_counter}/{total_runs}] "
                          f"{scenario_name} | p={prop} | {imputer.name}: "
                          f"{elapsed:.1f}s ({elapsed/N_SEEDS:.2f}s/seed)")

    if verbose:
        print(f"\nTotal runtime: {time.time() - t_start:.1f}s")

    results_df = pd.DataFrame(records)

    summary_df = (
        results_df
        .groupby(["scenario", "proportion", "method"], sort=False)
        .agg(
            rmse_mean=("rmse", "mean"),
            rmse_std=("rmse", "std"),
            mae_mean=("mae", "mean"),
            mae_std=("mae", "std"),
            avg_masked_cells=("n_masked_cells", "mean"),
            n_seeds=("seed", "count"),
        )
        .round(4)
        .reset_index()
    )

    return results_df, summary_df


In [ ]:
N_SEEDS = 1
PROPORTIONS = [0.30]
imputers = [GRAPEImputer(epochs=5000)]
results_df, summary_df = run_experiments(
    clean_data, imputers=imputers, scenario_filter=["MCAR"], verbose=False,
)
print(summary_df[["scenario", "proportion", "method", "rmse_mean"]])

['EGSAGE', 'EGSAGE', 'EGSAGE'] [True, True, True] [64]
total trainable_parameters:  26
train edge num is 233820, test edge num is input 233820, output 88180
epoch:  0
loss:  0.09256517142057419
test rmse:  0.15087127048960214
test l1:  0.148881196975708
epoch:  1
loss:  0.07815635949373245
test rmse:  0.12228043591427946
test l1:  0.11981921643018723
epoch:  2
loss:  0.06821956485509872
test rmse:  0.09538194713751452
test l1:  0.09221626818180084
epoch:  3
loss:  0.060352783650159836
test rmse:  0.07038251715293362
test l1:  0.06602364033460617
epoch:  4
loss:  0.054292481392621994
test rmse:  0.04761008230747479
test l1:  0.04088057205080986
epoch:  5
loss:  0.04976902902126312
test rmse:  0.02902881655577688
test l1:  0.016255374997854233
epoch:  6
loss:  0.04650547355413437
test rmse:  0.026319669582967928
test l1:  0.015805430710315704
epoch:  7
loss:  0.04447740316390991
test rmse:  0.04233058984032048
test l1:  0.03912029042840004
epoch:  8
loss:  0.04376103729009628
test rmse: 

In [17]:
print(summary_df)

  scenario  proportion method  rmse_mean  rmse_std  mae_mean  mae_std  \
0     MCAR         0.3  GRAPE     0.9743       NaN    0.6197      NaN   

   avg_masked_cells  n_seeds  
0           44090.0        1  
